# 03b Build Live Complexity Metrics

Aggregate Frankfurt live V2 Silver cellized flight states into PRU-inspired terminal complexity metrics and short-horizon horizontal hotspots.

Primary targets:

- `adsb_airspace_eddf.gld_airspace.horizontal_complexity_v2`
- `adsb_airspace_eddf.gld_airspace.horizontal_hotspots_v2`
- `adsb_airspace_eddf.gld_airspace.complexity_trend_v2`
- `adsb_airspace_eddf.obs.pipeline_run_log`


## V2 Complexity Contract

This notebook computes one live complexity run per `run_id + cell_scheme_id`.

- Spatial grain: `horizontal_cell_id`
- Time grain: `analysis_window_start`
- Features: `aircraft_count`, `heading_dispersion`, `speed_dispersion`, `active_vertical_cells`
- Hotspot stability rule: `minimum_active_windows`
- High-complexity threshold: `percentile_approx(complexity_score, high_complexity_percentile)`


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import yaml

from pyspark.sql import Window
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def parse_float(raw_value: str) -> float:
    return float(raw_value.strip())

def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())

def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {field.name: normalize_scalar(payload.get(field.name)) for field in target_schema}
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)

def format_for_id(value) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return str(numeric).replace('.', 'p')

def zscore_expr(column_name: str, *, mean_value, std_value):
    mean_number = 0.0 if mean_value is None else float(mean_value)
    if std_value is None or abs(float(std_value)) < 1e-9:
        return F.lit(0.0)
    return (F.col(column_name) - F.lit(mean_number)) / F.lit(float(std_value))

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_cell_scheme_id = ""
default_horizontal_cell_nm = "10"
default_vertical_cell_ft = "3000"
default_analysis_window_minutes = "15"
default_min_altitude_ft = "0"
default_max_altitude_ft = "24500"
default_projection_mode = "local_nm"
default_minimum_active_windows = "1"
default_high_complexity_percentile = "0.9"
default_overwrite_source_run = "true"

catalog = default_catalog
source_run_id_raw = default_source_run_id
cell_scheme_id_raw = default_cell_scheme_id
horizontal_cell_nm_raw = default_horizontal_cell_nm
vertical_cell_ft_raw = default_vertical_cell_ft
analysis_window_minutes_raw = default_analysis_window_minutes
min_altitude_ft_raw = default_min_altitude_ft
max_altitude_ft_raw = default_max_altitude_ft
projection_mode = default_projection_mode
minimum_active_windows_raw = default_minimum_active_windows
high_complexity_percentile_raw = default_high_complexity_percentile
overwrite_source_run_raw = default_overwrite_source_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("cell_scheme_id", default_cell_scheme_id)
    dbutils.widgets.text("horizontal_cell_nm", default_horizontal_cell_nm)
    dbutils.widgets.text("vertical_cell_ft", default_vertical_cell_ft)
    dbutils.widgets.text("analysis_window_minutes", default_analysis_window_minutes)
    dbutils.widgets.text("min_altitude_ft", default_min_altitude_ft)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("projection_mode", default_projection_mode, ["local_nm"])
    dbutils.widgets.text("minimum_active_windows", default_minimum_active_windows)
    dbutils.widgets.text("high_complexity_percentile", default_high_complexity_percentile)
    dbutils.widgets.dropdown("overwrite_source_run", default_overwrite_source_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    cell_scheme_id_raw = dbutils.widgets.get("cell_scheme_id").strip()
    horizontal_cell_nm_raw = dbutils.widgets.get("horizontal_cell_nm").strip() or default_horizontal_cell_nm
    vertical_cell_ft_raw = dbutils.widgets.get("vertical_cell_ft").strip() or default_vertical_cell_ft
    analysis_window_minutes_raw = dbutils.widgets.get("analysis_window_minutes").strip() or default_analysis_window_minutes
    min_altitude_ft_raw = dbutils.widgets.get("min_altitude_ft").strip() or default_min_altitude_ft
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    projection_mode = dbutils.widgets.get("projection_mode").strip() or default_projection_mode
    minimum_active_windows_raw = dbutils.widgets.get("minimum_active_windows").strip() or default_minimum_active_windows
    high_complexity_percentile_raw = dbutils.widgets.get("high_complexity_percentile").strip() or default_high_complexity_percentile
    overwrite_source_run_raw = dbutils.widgets.get("overwrite_source_run")

focus_airport = region_config["focus_airport"]
horizontal_cell_nm = parse_float(horizontal_cell_nm_raw)
vertical_cell_ft = parse_float(vertical_cell_ft_raw)
analysis_window_minutes = parse_int(analysis_window_minutes_raw)
min_altitude_ft = parse_float(min_altitude_ft_raw)
max_altitude_ft = parse_float(max_altitude_ft_raw)
minimum_active_windows = parse_int(minimum_active_windows_raw)
high_complexity_percentile = parse_float(high_complexity_percentile_raw)
overwrite_source_run = parse_bool(overwrite_source_run_raw)

if not 0 < high_complexity_percentile < 1:
    raise ValueError("high_complexity_percentile must be between 0 and 1")
if minimum_active_windows <= 0:
    raise ValueError("minimum_active_windows must be positive")

default_cell_scheme_id = (
    f"{focus_airport.lower()}_h{format_for_id(horizontal_cell_nm)}nm_"
    f"v{format_for_id(vertical_cell_ft)}ft_"
    f"w{analysis_window_minutes}m_"
    f"a{format_for_id(min_altitude_ft)}to{format_for_id(max_altitude_ft)}_"
    f"{projection_mode}_v2"
)
selected_cell_scheme_id = cell_scheme_id_raw.strip() or default_cell_scheme_id

silver_table_v2 = f"{catalog}.slv_airspace.flight_states_cellized_v2"
horizontal_complexity_table_v2 = f"{catalog}.gld_airspace.horizontal_complexity_v2"
horizontal_hotspots_table_v2 = f"{catalog}.gld_airspace.horizontal_hotspots_v2"
trend_table_v2 = f"{catalog}.gld_airspace.complexity_trend_v2"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"
source_run_id_input = source_run_id_raw.strip()

weight_defaults = {
    "aircraft_count": float(pipeline_config.get("complexity_weights", {}).get("aircraft_count", 0.25)),
    "heading_dispersion": float(pipeline_config.get("complexity_weights", {}).get("heading_dispersion", 0.25)),
    "speed_dispersion": float(pipeline_config.get("complexity_weights", {}).get("speed_dispersion", 0.25)),
    "active_vertical_cells": float(pipeline_config.get("complexity_weights", {}).get("altitude_dispersion", 0.25)),
}


In [ ]:
if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    latest_success_row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name = '02b_prepare_live_states_v2'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if latest_success_row is None:
        raise ValueError("No successful live V2 Silver run found. Run 02b_prepare_live_states_v2.ipynb first.")
    selected_source_run_id = latest_success_row["run_id"]

silver_df_v2 = (
    spark.table(silver_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)

source_row_count = silver_df_v2.count()
if source_row_count == 0:
    raise ValueError(
        f"No V2 Silver rows found for run_id={selected_source_run_id} and cell_scheme_id={selected_cell_scheme_id}."
    )

source_summary_row = silver_df_v2.agg(
    F.countDistinct("analysis_window_start").alias("window_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.countDistinct("horizontal_cell_id").alias("horizontal_cell_count"),
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time")
).first()

run_plan = {
    "catalog": catalog,
    "focus_airport": focus_airport,
    "source_run_id": selected_source_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "silver_table_v2": silver_table_v2,
    "horizontal_complexity_table_v2": horizontal_complexity_table_v2,
    "horizontal_hotspots_table_v2": horizontal_hotspots_table_v2,
    "trend_table_v2": trend_table_v2,
    "source_row_count": source_row_count,
    "source_window_count": int(source_summary_row["window_count"]),
    "source_aircraft_count": int(source_summary_row["aircraft_count"]),
    "source_horizontal_cell_count": int(source_summary_row["horizontal_cell_count"]),
    "minimum_active_windows": minimum_active_windows,
    "high_complexity_percentile": high_complexity_percentile,
    "complexity_weights": weight_defaults,
    "overwrite_source_run": overwrite_source_run,
}

run_plan


In [ ]:
horizontal_metrics_base_df = (
    silver_df_v2
    .groupBy("analysis_window_start", "horizontal_cell_id")
    .agg(
        F.countDistinct("icao24").alias("aircraft_count"),
        F.avg(F.cos(F.radians(F.col("heading_deg")))).alias("mean_heading_cos"),
        F.avg(F.sin(F.radians(F.col("heading_deg")))).alias("mean_heading_sin"),
        F.count(F.col("heading_deg")).alias("heading_obs_count"),
        F.stddev_samp("speed_kt").alias("speed_dispersion_raw"),
        F.countDistinct("vertical_cell_id").alias("active_vertical_cells")
    )
    .withColumn(
        "heading_dispersion",
        F.when(F.col("heading_obs_count") <= 1, F.lit(0.0)).otherwise(
            F.lit(1.0) - F.sqrt(
                F.pow(F.coalesce(F.col("mean_heading_cos"), F.lit(0.0)), F.lit(2.0)) +
                F.pow(F.coalesce(F.col("mean_heading_sin"), F.lit(0.0)), F.lit(2.0))
            )
        )
    )
    .withColumn("speed_dispersion", F.coalesce(F.col("speed_dispersion_raw"), F.lit(0.0)))
)

metric_stats_row = horizontal_metrics_base_df.agg(
    F.avg("aircraft_count").alias("aircraft_count_mean"),
    F.stddev_samp("aircraft_count").alias("aircraft_count_std"),
    F.avg("heading_dispersion").alias("heading_dispersion_mean"),
    F.stddev_samp("heading_dispersion").alias("heading_dispersion_std"),
    F.avg("speed_dispersion").alias("speed_dispersion_mean"),
    F.stddev_samp("speed_dispersion").alias("speed_dispersion_std"),
    F.avg("active_vertical_cells").alias("active_vertical_cells_mean"),
    F.stddev_samp("active_vertical_cells").alias("active_vertical_cells_std")
).first()

horizontal_complexity_df_v2 = (
    horizontal_metrics_base_df
    .withColumn("z_aircraft_count", zscore_expr("aircraft_count", mean_value=metric_stats_row["aircraft_count_mean"], std_value=metric_stats_row["aircraft_count_std"]))
    .withColumn("z_heading_dispersion", zscore_expr("heading_dispersion", mean_value=metric_stats_row["heading_dispersion_mean"], std_value=metric_stats_row["heading_dispersion_std"]))
    .withColumn("z_speed_dispersion", zscore_expr("speed_dispersion", mean_value=metric_stats_row["speed_dispersion_mean"], std_value=metric_stats_row["speed_dispersion_std"]))
    .withColumn("z_active_vertical_cells", zscore_expr("active_vertical_cells", mean_value=metric_stats_row["active_vertical_cells_mean"], std_value=metric_stats_row["active_vertical_cells_std"]))
    .withColumn(
        "complexity_score_raw",
        weight_defaults["aircraft_count"] * F.col("z_aircraft_count") +
        weight_defaults["heading_dispersion"] * F.col("z_heading_dispersion") +
        weight_defaults["speed_dispersion"] * F.col("z_speed_dispersion") +
        weight_defaults["active_vertical_cells"] * F.col("z_active_vertical_cells")
    )
    .select(
        F.col("analysis_window_start").alias("window_start"),
        F.col("horizontal_cell_id"),
        F.col("aircraft_count").cast("long").alias("aircraft_count"),
        F.round(F.col("heading_dispersion"), 6).alias("heading_dispersion"),
        F.round(F.col("speed_dispersion"), 6).alias("speed_dispersion"),
        F.col("active_vertical_cells").cast("long").alias("active_vertical_cells"),
        F.round(F.col("complexity_score_raw"), 6).alias("complexity_score"),
        F.current_timestamp().alias("computed_at"),
        F.lit(selected_cell_scheme_id).alias("cell_scheme_id"),
        F.lit(selected_source_run_id).alias("run_id"),
    )
)

horizontal_quality_row = horizontal_complexity_df_v2.agg(
    F.count("*").alias("horizontal_window_count"),
    F.countDistinct("horizontal_cell_id").alias("horizontal_cell_count"),
    F.countDistinct("window_start").alias("window_count"),
    F.min("complexity_score").alias("min_complexity_score"),
    F.max("complexity_score").alias("max_complexity_score")
).first()

if int(horizontal_quality_row["horizontal_window_count"]) == 0:
    raise ValueError("V2 complexity aggregation produced no rows. Check the cellized Silver input.")

hotspot_threshold = float(
    horizontal_complexity_df_v2.select(
        F.expr(f"percentile_approx(complexity_score, {high_complexity_percentile})").alias("threshold")
    ).first()["threshold"] or 0.0
)

hotspots_base_df = (
    horizontal_complexity_df_v2
    .groupBy("horizontal_cell_id")
    .agg(
        F.count("*").alias("active_windows"),
        F.round(F.avg("complexity_score"), 6).alias("avg_complexity_score"),
        F.round(F.max("complexity_score"), 6).alias("peak_complexity_score"),
        F.sum(F.when(F.col("complexity_score") >= F.lit(hotspot_threshold), F.lit(1)).otherwise(F.lit(0))).cast("long").alias("num_high_complexity_windows")
    )
    .where(F.col("active_windows") >= F.lit(minimum_active_windows))
)

ranking_window = Window.orderBy(
    F.col("avg_complexity_score").desc(),
    F.col("peak_complexity_score").desc(),
    F.col("horizontal_cell_id").asc(),
)

horizontal_hotspots_df_v2 = (
    hotspots_base_df
    .withColumn("ranking", F.dense_rank().over(ranking_window).cast("int"))
    .withColumn("computed_at", F.current_timestamp())
    .withColumn("cell_scheme_id", F.lit(selected_cell_scheme_id))
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        "horizontal_cell_id",
        "active_windows",
        "avg_complexity_score",
        "peak_complexity_score",
        "num_high_complexity_windows",
        "ranking",
        "computed_at",
        "cell_scheme_id",
        "run_id",
    )
)

aircraft_trend_df_v2 = silver_df_v2.groupBy("analysis_window_start").agg(
    F.countDistinct("icao24").alias("active_aircraft_count")
)

complexity_trend_df_v2 = (
    horizontal_complexity_df_v2
    .groupBy("window_start")
    .agg(
        F.round(F.avg("complexity_score"), 6).alias("avg_complexity_score"),
        F.round(F.max("complexity_score"), 6).alias("peak_complexity_score"),
        F.countDistinct("horizontal_cell_id").alias("active_horizontal_cell_count")
    )
    .join(aircraft_trend_df_v2, horizontal_complexity_df_v2.window_start == aircraft_trend_df_v2.analysis_window_start, how="left")
    .drop("analysis_window_start")
    .withColumn("computed_at", F.current_timestamp())
    .withColumn("cell_scheme_id", F.lit(selected_cell_scheme_id))
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        "window_start",
        "avg_complexity_score",
        "peak_complexity_score",
        F.col("active_horizontal_cell_count").cast("long").alias("active_horizontal_cell_count"),
        F.col("active_aircraft_count").cast("long").alias("active_aircraft_count"),
        "computed_at",
        "cell_scheme_id",
        "run_id",
    )
)

hotspots_count = horizontal_hotspots_df_v2.count()
trend_count = complexity_trend_df_v2.count()

metric_summary = {
    "source_run_id": selected_source_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "source_row_count": source_row_count,
    "horizontal_window_count": int(horizontal_quality_row["horizontal_window_count"]),
    "horizontal_cell_count": int(horizontal_quality_row["horizontal_cell_count"]),
    "window_count": int(horizontal_quality_row["window_count"]),
    "hotspots_count": int(hotspots_count),
    "trend_count": int(trend_count),
    "hotspot_threshold": hotspot_threshold,
    "min_complexity_score": float(horizontal_quality_row["min_complexity_score"]),
    "max_complexity_score": float(horizontal_quality_row["max_complexity_score"]),
}

metric_summary


In [ ]:
pipeline_started_at = utc_now_naive()
pipeline_status = "success"
pipeline_error_message = None

try:
    if overwrite_source_run:
        spark.sql(f"DELETE FROM {horizontal_complexity_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{selected_cell_scheme_id}'")
        spark.sql(f"DELETE FROM {horizontal_hotspots_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{selected_cell_scheme_id}'")
        spark.sql(f"DELETE FROM {trend_table_v2} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{selected_cell_scheme_id}'")

    horizontal_complexity_df_v2.write.mode("append").saveAsTable(horizontal_complexity_table_v2)
    horizontal_hotspots_df_v2.write.mode("append").saveAsTable(horizontal_hotspots_table_v2)
    complexity_trend_df_v2.write.mode("append").saveAsTable(trend_table_v2)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=pipeline_log_table,
        payload={
            "run_id": selected_source_run_id,
            "pipeline_name": "03b_build_live_complexity_metrics",
            "layer": "gold_live_v2",
            "target_table": horizontal_complexity_table_v2,
            "status": pipeline_status,
            "rows_read": source_row_count,
            "rows_written": (
                metric_summary["horizontal_window_count"] + metric_summary["hotspots_count"] + metric_summary["trend_count"]
                if pipeline_status == "success" else 0
            ),
            "started_at": pipeline_started_at,
            "completed_at": utc_now_naive(),
            "error_message": pipeline_error_message,
            "metadata_json": json.dumps(
                {
                    "cell_scheme_id": selected_cell_scheme_id,
                    "minimum_active_windows": minimum_active_windows,
                    "high_complexity_percentile": high_complexity_percentile,
                    "weights": weight_defaults,
                    "source_pipeline_name": "02b_prepare_live_states_v2",
                },
                sort_keys=True,
                default=str,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "source_run_id": selected_source_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "source_row_count": source_row_count,
    "horizontal_complexity_table_v2": horizontal_complexity_table_v2,
    "horizontal_hotspots_table_v2": horizontal_hotspots_table_v2,
    "trend_table_v2": trend_table_v2,
    "horizontal_window_count": metric_summary["horizontal_window_count"],
    "hotspots_count": metric_summary["hotspots_count"],
    "trend_count": metric_summary["trend_count"],
    "hotspot_threshold": metric_summary["hotspot_threshold"],
    "min_complexity_score": metric_summary["min_complexity_score"],
    "max_complexity_score": metric_summary["max_complexity_score"],
}

final_summary
